# AgentCore 메모리 브랜칭을 활용한 Strands 멀티 에이전트 병렬 실행 시스템

## 소개

이 노트북은 **AgentCore 메모리 브랜칭** - 여러 전문 에이전트가 공통 메모리 리소스를 공유하면서 격리된 메모리 컨텍스트를 유지할 수 있게 하는 강력한 기능을 보여줍니다. 이는 멀티 에이전트 시스템, 특히 Strands Agent Graph와 같은 병렬 실행 패턴을 사용하는 시스템에 필수적입니다.

## 메모리 브랜칭이 중요한 이유

멀티 에이전트 시스템에서 다른 에이전트들은 종종 다음이 필요합니다:
- **별도의 대화 컨텍스트 유지** - 각 에이전트가 간섭 없이 자신의 도메인에 집중
- **병렬 실행** - 여러 에이전트가 메모리 충돌 없이 동시에 작업 가능
- **공통 세션 공유** - 모든 에이전트가 컨텍스트를 격리하면서 동일한 사용자 세션에 기여
- **관련 기록 접근** - 에이전트가 컨텍스트를 혼합하지 않고 자신의 과거 상호작용을 검색 가능

AgentCore 메모리 브랜칭은 코드를 위한 Git 브랜치와 유사하게, 단일 메모리 세션 내에서 여러 대화 브랜치를 허용하여 이러한 과제를 해결합니다.

## 튜토리얼 세부 정보

| 항목                | 세부 내용                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 메모리 브랜칭을 활용한 멀티 에이전트                                             |
| 에이전트 유스케이스 | 여행 계획 어시스턴트                                                             |
| 에이전트 프레임워크 | Strands Agent Graph (병렬 실행 지원)                                             |
| LLM 모델            | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성 요소  | AgentCore 메모리 브랜칭, Strands 멀티 에이전트 그래프, 병렬 실행                 |
| 난이도              | 중급                                                                             |


학습 내용:

- 다른 에이전트를 위한 메모리 브랜치를 생성하고 관리하는 방법
- 멀티 에이전트 아키텍처에서 격리된 메모리 컨텍스트 구현
- 병렬 실행을 지원하는 Strands 에이전트 그래프 구축
- 브랜칭이 안전한 동시 메모리 접근을 가능하게 하는 방법
- 브랜치별 대화 기록 조회 및 검사

### 시나리오 맥락

각각 자체 메모리 브랜치를 가진 3개의 에이전트로 **여행 계획 시스템**을 구축합니다:
1. **여행 코디네이터** (main 브랜치) - 전체 여행 계획을 조율
2. **항공편 예약 어시스턴트** (flight_agent_memory 브랜치) - 항공 여행 쿼리 처리
3. **호텔 예약 어시스턴트** (hotel_agent_memory 브랜치) - 숙박 요청 관리

코디네이터는 병렬로 실행되는 전문 에이전트에 위임할 수 있으며, 각 에이전트는 메모리 브랜칭을 통해 자체 대화 기록을 유지합니다.

## 아키텍처
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## 사전 요구사항
- Python 3.10+
- 적절한 권한이 있는 AWS 계정
- AgentCore 메모리에 대한 적절한 권한이 있는 AWS IAM 역할
- Amazon Bedrock 모델 접근

환경을 설정하고 브랜칭을 지원하는 공유 메모리 리소스를 생성하는 것부터 시작하겠습니다!

## 1단계: 환경 설정
이 노트북을 실행하는 데 필요한 모든 라이브러리를 가져오고 클라이언트를 정의하는 것부터 시작하겠습니다.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import logging
from datetime import datetime
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent

Amazon Bedrock 모델 및 AgentCore에 대한 적절한 권한이 있는 리전과 역할을 정의합니다.

In [ ]:
import os
region = os.getenv('AWS_REGION', 'us-west-2')
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("agentcore-memory")

## 2단계: 브랜칭을 지원하는 공유 메모리 생성

여러 브랜치를 지원하는 단일 메모리 리소스를 생성합니다 - 각 에이전트에 하나씩. 이 공유 메모리 리소스는 기반 역할을 하고, 브랜치는 각 에이전트의 대화를 위한 격리된 컨텍스트를 제공합니다.

Git 저장소처럼 생각하세요: 하나의 저장소(메모리 리소스)에 여러 브랜치(에이전트 컨텍스트)가 있습니다.

In [ ]:
from bedrock_agentcore.memory import MemoryClient

In [ ]:
client = MemoryClient(region_name=region)
memory_name = "TravelAgent_STM_%s" % datetime.now().strftime("%Y%m%d%H%M%S")
memory_id = None


In [ ]:
from botocore.exceptions import ClientError

try:
    print("메모리 생성 중...")
    memory_name = memory_name

    # 메모리 리소스 생성
    memory = client.create_memory_and_wait(
        name=memory_name,                       # 이 메모리 저장소의 고유 이름
        description="Travel Agent STM",         # 사람이 읽을 수 있는 설명
        strategies=[],                          # 단기 메모리를 위한 특별한 전략 없음
        event_expiry_days=7,                    # 7일 후 메모리 만료
        max_wait=300,                           # 메모리 생성 대기 최대 시간 (5분)
        poll_interval=10                        # 10초마다 상태 확인
    )

    # 메모리 ID 추출 및 출력
    memory_id = memory['id']
    print(f"메모리가 성공적으로 생성되었습니다. ID: {memory_id}")
except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # 메모리가 이미 존재하면 해당 ID를 가져옴
        memories = client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"메모리가 이미 존재합니다. 기존 메모리 ID 사용: {memory_id}")
except Exception as e:
    # 메모리 생성 중 오류 처리
    print(f"오류: {e}")
    import traceback
    traceback.print_exc()

    # 오류 시 정리 - 부분적으로 생성된 메모리 삭제
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"메모리 정리 완료: {memory_id}")
        except Exception as cleanup_error:
            logger.info(f"메모리 정리 실패: {cleanup_error}")

### 멀티 에이전트 시스템을 위한 메모리 브랜칭 이해

우리가 생성한 메모리 리소스는 **브랜칭**을 지원합니다 - 멀티 에이전트 아키텍처의 핵심 기능입니다. 작동 방식은 다음과 같습니다:

**단일 메모리 리소스, 다중 브랜치:**
- 모든 에이전트가 동일한 `memory_id`와 `session_id`를 공유
- 각 에이전트는 격리된 컨텍스트를 위해 자체 `branch_name`을 가짐

**멀티 에이전트 시스템의 주요 이점:**

1. **컨텍스트 격리**: 각 에이전트가 간섭 없이 자체 대화 기록을 유지
   - 항공편 에이전트는 항공편 관련 대화만 봄
   - 호텔 에이전트는 호텔 관련 대화만 봄
   - 코디네이터는 메인 조율 흐름을 봄

2. **병렬 실행 안전성**: 여러 에이전트가 동시에 실행 가능
   - 에이전트가 병렬로 실행될 때 메모리 충돌 없음
   - 각 브랜치에 독립적으로 접근 가능
   - 동시 실행을 지원하는 Strands Agent Graph에 필수적

3. **명확한 감사 추적**: 각 에이전트의 상호작용을 추적 가능
   - 각 에이전트가 무엇을 논의했는지 검사
   - 에이전트별 문제 디버그
   - 멀티 에이전트 대화 흐름 이해

## 3단계: 브랜치 지원이 포함된 메모리 훅 프로바이더 생성

`ShortTermMemoryHook` 클래스는 브랜치 인식 메모리 관리를 구현합니다. 이것이 멀티 에이전트 시스템에서 메모리 브랜칭을 가능하게 하는 핵심 구성 요소입니다.

**주요 기능:**

1. **브랜치 초기화**: 각 에이전트에 대한 브랜치를 자동으로 생성
   - 코디네이터 에이전트를 위한 main 브랜치
   - 서브 에이전트를 위한 전문 브랜치 (예: `flight_agent_memory`, `hotel_agent_memory`)
   - 메인 대화 타임라인에서 포크된 브랜치

2. **브랜치별 메모리 검색**: 각 에이전트가 자신의 컨텍스트만 로드
   - `on_agent_initialized()`가 에이전트의 브랜치에서 대화 기록을 가져옴
   - 에이전트 간 컨텍스트 오염 방지
   - 에이전트가 집중된, 도메인 특화 대화를 유지할 수 있게 함

3. **브랜치별 메모리 저장**: 대화가 올바른 브랜치에 저장됨
   - `on_message_added()`가 에이전트의 지정된 브랜치에 메시지를 저장
   - 병렬 에이전트 실행의 동시 쓰기 지원
   - 경쟁 조건이나 메모리 충돌 없음

이 훅 프로바이더가 AgentCore 메모리로 병렬 에이전트 실행을 안전하고 효율적으로 만들어줍니다.

In [ ]:
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole
from bedrock_agentcore.memory import MemorySessionManager
class ShortTermMemoryHook(HookProvider):
    def __init__(self, memory_id: str, region_name: str = "us-west-2", branch_name: str = "main"):
        """Initialize the hook with a MemorySessionManager.

        Args:
            memory_id: The AgentCore Memory ID
            region_name: AWS region for the memory service
            branch_name: Branch name for this agent's memory (default: "main")
        """
        self.memory_manager = MemorySessionManager(
            memory_id=memory_id,
            region_name=region_name
        )
        self.memory_id = memory_id
        self.branch_name = branch_name
        self._sessions = {}  # 액터/세션 조합별 세션 객체 캐시
        self._branch_initialized = False  # 브랜치 생성 여부 추적

    def _get_or_create_session(self, actor_id: str, session_id: str):
        """Get or create a MemorySession for the given actor/session.

        Args:
            actor_id: The actor identifier
            session_id: The session identifier

        Returns:
            MemorySession object
        """
        key = f"{actor_id}:{session_id}"
        if key not in self._sessions:
            self._sessions[key] = self.memory_manager.create_memory_session(
                actor_id=actor_id,
                session_id=session_id
            )
        return self._sessions[key]

    def _initialize_branch(self, actor_id: str, session_id: str):
        """Initialize a branch if it doesn't exist and this is not the main branch.

        Args:
            actor_id: The actor identifier
            session_id: The session identifier
        """
        if self._branch_initialized or self.branch_name == "main":
            return

        try:
            memory_session = self._get_or_create_session(actor_id, session_id)

            # 브랜치가 이미 존재하는지 확인
            branches = memory_session.list_branches()
            branch_exists = any(b.name == self.branch_name for b in branches)

            if not branch_exists:
                # 포크할 main 브랜치의 마지막 이벤트 가져오기
                main_events = memory_session.list_events(branch_name="main")
                if main_events:
                    last_event = main_events[-1]
                    # 초기 메시지와 함께 브랜치 생성
                    memory_session.fork_conversation(
                        root_event_id=last_event.eventId,
                        branch_name=self.branch_name,
                        messages=[
                            ConversationalMessage(f"Starting {self.branch_name} branch", MessageRole.ASSISTANT)
                        ]
                    )
                    logger.info(f"브랜치 생성 완료: {self.branch_name}")

            self._branch_initialized = True

        except Exception as e:
            logger.error(f"브랜치 {self.branch_name} 초기화 실패: {e}", exc_info=True)

    def on_agent_initialized(self, event: AgentInitializedEvent):
        """에이전트 시작 시 최근 대화 기록 로드"""
        try:
            # 에이전트 상태에서 세션 정보 가져오기
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")

            if not actor_id or not session_id:
                logger.warning("에이전트 상태에 actor_id 또는 session_id가 없습니다")
                return

            # 메모리 세션 가져오기
            memory_session = self._get_or_create_session(actor_id, session_id)

            # main이 아닌 브랜치의 경우, main 브랜치에 이벤트가 있으면 초기화
            if self.branch_name != "main":
                try:
                    main_events = memory_session.list_events(branch_name="main")
                    if len(main_events) > 0:
                        self._initialize_branch(actor_id, session_id)
                except Exception as e:
                    # 첫 번째 호출 시 main 브랜치가 아직 존재하지 않을 수 있음
                    logger.info(f"main 브랜치가 아직 없습니다. {self.branch_name} 브랜치는 나중에 초기화됩니다: {e}")

            # 턴을 가져오기 전에 브랜치가 존재하는지 확인
            branches = memory_session.list_branches()
            branch_exists = any(b.name == self.branch_name for b in branches)
            
            recent_turns = []
            if branch_exists:
                # 브랜치가 존재할 때만 턴 가져오기
                recent_turns = memory_session.get_last_k_turns(
                    k=5,
                    branch_name=self.branch_name
                )
            else:
                logger.info(f"브랜치 '{self.branch_name}'가 아직 존재하지 않습니다. 턴 검색을 건너뜁니다")

            if len(recent_turns) > 0:
                # 컨텍스트를 위한 대화 기록 포맷팅
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        role = message.get('role', 'unknown').lower()
                        text = message.get('content', {}).get('text', '')
                        if text:
                            context_messages.append(f"{role.title()}: {text}")

                if context_messages:
                    context = "\n".join(context_messages)
                    logger.info(f"브랜치 '{self.branch_name}'에서 컨텍스트 로드 ({len(context_messages)}개 메시지)")

                    # 에이전트의 시스템 프롬프트에 컨텍스트 추가
                    event.agent.system_prompt += (
                        f"\n\nRecent conversation history (from {self.branch_name}):\n{context}\n\n"
                        "Continue the conversation naturally based on this context."
                    )

                    logger.info(f"브랜치 '{self.branch_name}'에서 {len(recent_turns)}개의 최근 대화 턴 로드 완료")
            else:
                logger.info(f"브랜치 '{self.branch_name}'에 이전 대화 기록이 없습니다")

        except Exception as e:
            logger.error(f"대화 기록 로드 실패: {e}", exc_info=True)

    def on_message_added(self, event: MessageAddedEvent):
        """적절한 브랜치에 대화 턴 저장"""
        try:
            # 에이전트 상태에서 세션 정보 가져오기
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")

            if not actor_id or not session_id:
                logger.warning("에이전트 상태에 actor_id 또는 session_id가 없습니다")
                return

            # 메모리 세션 가져오기
            memory_session = self._get_or_create_session(actor_id, session_id)

            # 마지막 메시지 가져오기
            messages = event.agent.messages
            if not messages:
                return

            last_message = messages[-1]
            role_str = last_message.get("role", "").upper()
            content_text = last_message.get("content", [{}])[0].get("text", "")

            if not content_text:
                logger.debug("빈 메시지 건너뜀")
                return

            # 역할 문자열을 MessageRole enum으로 매핑
            role_mapping = {
                "USER": MessageRole.USER,
                "ASSISTANT": MessageRole.ASSISTANT,
                "TOOL": MessageRole.TOOL,
            }
            message_role = role_mapping.get(role_str, MessageRole.USER)

            # 적절한 브랜치에 메시지 저장
            if self.branch_name == "main":
                # main 브랜치 - 일반적으로 턴 추가
                memory_session.add_turns(
                    messages=[ConversationalMessage(content_text, message_role)]
                )
            else:
                # main이 아닌 브랜치 - 기존 브랜치에 추가 필요
                # 브랜치가 존재하지 않으면 초기화
                if not self._branch_initialized:
                    self._initialize_branch(actor_id, session_id)

                # 이 브랜치의 최신 이벤트 가져오기
                branch_events = memory_session.list_events(branch_name=self.branch_name)
                if branch_events:
                    # 브랜치 이름을 지정하여 기존 브랜치에 추가 (rootEventId 없이)
                    memory_session.add_turns(
                        messages=[ConversationalMessage(content_text, message_role)],
                        branch={"name": self.branch_name}
                    )
                else:
                    # _initialize_branch가 작동했다면 발생하지 않지만, 처리
                    logger.warning(f"초기화 후 브랜치 {self.branch_name}를 찾을 수 없습니다")
                    self._initialize_branch(actor_id, session_id)

            logger.debug(f"브랜치 '{self.branch_name}'에 메시지 저장 완료: {role_str}")

        except Exception as e:
            logger.error(f"메시지 저장 실패: {e}", exc_info=True)

    def create_branch(self, actor_id: str, session_id: str,
                      root_event_id: str, branch_name: str,
                      messages: list):
        """Create a new conversation branch.

        Args:
            actor_id: The actor identifier
            session_id: The session identifier
            root_event_id: Event ID to branch from
            branch_name: Name for the new branch
            messages: List of ConversationalMessage objects to add to the branch
        """
        memory_session = self._get_or_create_session(actor_id, session_id)
        return memory_session.fork_conversation(
            root_event_id=root_event_id,
            branch_name=branch_name,
            messages=messages
        )

    def list_branches(self, actor_id: str, session_id: str):
        """List all branches for a session.

        Args:
            actor_id: The actor identifier
            session_id: The session identifier

        Returns:
            List of branch information
        """
        memory_session = self._get_or_create_session(actor_id, session_id)
        return memory_session.list_branches()

    def get_session(self, actor_id: str, session_id: str):
        """Get the memory session object for direct access.

        Args:
            actor_id: The actor identifier
            session_id: The session identifier

        Returns:
            MemorySession object
        """
        return self._get_or_create_session(actor_id, session_id)

    def register_hooks(self, registry: HookRegistry) -> None:
        """Register memory hooks with the registry.

        Args:
            registry: The HookRegistry to register callbacks with
        """
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)

## 4단계: Strands Agent Graph로 멀티 에이전트 아키텍처 생성

이제 **Strands Agent Graph** - 병렬 에이전트 실행을 지원하는 프레임워크를 사용하여 멀티 에이전트 시스템을 구축합니다. 각 에이전트는 자체 메모리 브랜치로 구성되어 안전한 동시 운영이 가능합니다.

**아키텍처 개요:**
- **코디네이터 에이전트** → `main` 브랜치 사용
- **항공편 에이전트** → `flight_agent_memory` 브랜치 사용
- **호텔 에이전트** → `hotel_agent_memory` 브랜치 사용

모든 에이전트는 동일한 `session_id`를 공유하지만 브랜칭을 통해 격리된 대화 컨텍스트를 유지합니다.

In [ ]:
# 필요한 구성 요소 가져오기
from strands import Agent, tool

In [ ]:
# 각 전문 에이전트에 대해 고유한 액터 ID를 생성하되 세션 ID는 공유
actor_id = f"travel-user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
session_id = f"travel-session-{datetime.now().strftime('%Y%m%d%H%M%S')}"
namespace = f"travel/{actor_id}/preferences/"

### 브랜치별 메모리를 가진 전문 에이전트 생성

전문 에이전트를 위한 시스템 프롬프트를 정의합니다. 각 에이전트는 자체 메모리 브랜치로 구성되어 대화 격리와 병렬 실행이 가능합니다.

In [ ]:
# 호텔 예약 전문가의 시스템 프롬프트
HOTEL_BOOKING_PROMPT = f"""You are a hotel booking assistant. Help customers find hotels, make reservations, and answer questions about accommodations and amenities. 
Provide clear information about availability, pricing, and booking procedures in a friendly, helpful manner."""

# 항공편 예약 전문가의 시스템 프롬프트
FLIGHT_BOOKING_PROMPT = f"""You are a flight booking assistant. Help customers find flights, make reservations, and answer questions about airlines, routes, and travel policies. 
Provide clear information about flight availability, pricing, schedules, and booking procedures in a friendly, helpful manner."""

In [ ]:
flight_memory_hooks = None
hotel_memory_hooks = None

### 메모리 브랜칭을 활용한 에이전트 구현

각 전문 에이전트는 다음으로 구성됩니다:
- 격리된 메모리 컨텍스트를 위한 고유한 `branch_name`
- 공유 세션 관리를 위한 동일한 `memory_id` 및 `session_id`
- 브랜치별 작업을 처리하는 `ShortTermMemoryHook`

**주요 구현 세부 사항:**
- `flight_booking_agent()`는 브랜치 `flight_agent_memory` 사용
- `hotel_booking_agent()`는 브랜치 `hotel_agent_memory` 사용

In [ ]:
def flight_booking_agent() -> Agent:

    global flight_memory_hooks
    try:
        if flight_memory_hooks is None:
            # "flight_agent_memory" 브랜치 이름으로 훅 생성
            flight_memory_hooks = ShortTermMemoryHook(
                memory_id=memory_id,
                region_name=region,
                branch_name="flight_agent_memory"
            )

        flight_agent = Agent(
            hooks=[flight_memory_hooks],
            model=MODEL_ID,
            system_prompt=FLIGHT_BOOKING_PROMPT,
            state={"actor_id": actor_id, "session_id": session_id}
        )

        
        return flight_agent
    except Exception as e:
        return f"Error in flight booking assistant: {str(e)}"

def hotel_booking_agent() -> Agent:

    global hotel_memory_hooks
    try:
        if hotel_memory_hooks is None:
            # "hotel_agent_memory" 브랜치 이름으로 훅 생성
            hotel_memory_hooks = ShortTermMemoryHook(
                memory_id=memory_id,
                region_name=region,
                branch_name="hotel_agent_memory"
            )

        hotel_booking_agent = Agent(
            hooks=[hotel_memory_hooks],
            model=MODEL_ID,
            system_prompt=HOTEL_BOOKING_PROMPT,
            state={"actor_id": actor_id, "session_id": session_id}
        )

        return hotel_booking_agent
    except Exception as e:
        return f"Error in hotel booking assistant: {str(e)}"

### 코디네이터 에이전트 생성

코디네이터 에이전트는 `main` 브랜치(기본값)를 사용하고 전문 에이전트를 조율합니다. Strands Agent Graph를 사용할 때 병렬로 실행될 수 있는 항공편 및 호텔 에이전트에 작업을 위임할 수 있습니다.

In [ ]:
# 코디네이터 에이전트의 시스템 프롬프트
TRAVEL_AGENT_SYSTEM_PROMPT = """
You are a comprehensive travel planning assistant that coordinates between specialized tools:
- For flight-related queries (bookings, schedules, airlines, routes) → Use the flight_booking_agent
- For hotel-related queries (accommodations, amenities, reservations) → Use the hotel_booking_agent
- For complete travel packages → Use both tools as needed to provide comprehensive information
- For general travel advice or simple travel questions → Answer directly

Each agent will have its own memory in case the user asks about historic data.
When handling complex travel requests, coordinate information from both tools to create a cohesive travel plan.
Provide clear organization when presenting information from multiple sources. \
Ask max two questions per turn. Keep the messages short, don't overwhelm the customer.
"""

In [ ]:
def travel_booking_agent() -> Agent:

    agent_memory_hooks = ShortTermMemoryHook(
                    memory_id=memory_id,
                    region_name=region,
                )
    travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    hooks=[agent_memory_hooks],
    model=MODEL_ID,
    state={
        "actor_id": actor_id,
        "session_id": session_id
        }
    )

    return travel_agent

### 병렬 실행을 지원하는 에이전트 그래프 구축

이제 에이전트를 **Strands Agent Graph**로 조립합니다. 이 그래프 구조는 다음을 가능하게 합니다:

**병렬 실행:**
- 코디네이터가 항공편과 호텔 정보를 모두 필요로 할 때, 두 에이전트가 동시에 실행 가능
- 메모리 브랜칭이 동시 실행 중 충돌 방지
- 각 에이전트가 자체 브랜치에서 독립적으로 읽기/쓰기

**메모리 브랜치 매핑:**
```
Session: travel-session-xxx
├── main branch              → 여행 코디네이터
├── flight_agent_memory      → 항공편 예약 에이전트
└── hotel_agent_memory       → 호텔 예약 에이전트
```

**이것이 중요한 이유:**
- 브랜칭 없이는 병렬 에이전트가 서로의 메모리를 덮어쓸 수 있음
- 브랜칭을 사용하면 각 에이전트가 자체 대화 스레드를 유지
- 코디네이터가 안전하게 여러 에이전트에 동시에 위임 가능

In [ ]:
import logging
from strands import Agent
from strands.multiagent import GraphBuilder

# 디버그 로그 활성화 및 stderr로 출력
logging.getLogger("strands.multiagent").setLevel(logging.DEBUG)
logging.basicConfig(
    format="%(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()]
)

# Strands Agent Graph 구축
# 이 그래프 구조는 전문 에이전트의 병렬 실행을 가능하게 합니다
# 메모리 브랜칭이 충돌 없이 안전한 동시 접근을 보장합니다
builder = GraphBuilder()

# 노드 추가 - 각 에이전트는 자체 메모리 브랜치를 가짐
builder.add_node(travel_booking_agent(), "travel_agent")           # 'main' 브랜치 사용
builder.add_node(flight_booking_agent(), "flight_booking_agent")   # 'flight_agent_memory' 브랜치 사용
builder.add_node(hotel_booking_agent(), "hotel_booking_agent")     # 'hotel_agent_memory' 브랜치 사용

# 엣지 추가 - 코디네이터가 위임할 수 있는 에이전트 정의
# 그래프는 두 에이전트가 모두 필요할 때 항공편과 호텔 에이전트를 병렬로 실행 가능
builder.add_edge("travel_agent", "flight_booking_agent")
builder.add_edge("travel_agent", "hotel_booking_agent")

# 진입점 설정 - 코디네이터 에이전트가 먼저 사용자 입력을 받음
builder.set_entry_point("travel_agent")

# 안전을 위한 실행 제한 구성
builder.set_execution_timeout(600)   # 10분 타임아웃

# 그래프 빌드 - 격리된 메모리 컨텍스트로 병렬 실행 준비 완료
graph = builder.build()


### 메모리 브랜칭을 활용한 멀티 에이전트 시스템 준비 완료!

에이전트 그래프가 다음으로 구성되었습니다:
- 격리된 메모리 브랜치를 가진 3개의 에이전트
- Strands Agent Graph를 통한 병렬 실행 기능
- AgentCore 메모리 브랜칭을 통한 안전한 동시 메모리 접근
- 자동 브랜치 생성 및 관리

## 멀티 에이전트 시스템 테스트

여러 에이전트를 트리거할 여행 계획 시나리오로 테스트해 보겠습니다:

In [ ]:
# LA에서 마드리드까지 7월 1일부터 8월 2일까지 여행 예약 요청
response = graph("Hello, I would like to book a trip from LA to Madrid. From July 1 to August 2.")

## 메모리 브랜치 검사

AgentCore 메모리 브랜칭의 주요 장점 중 하나는 각 에이전트의 대화 기록을 독립적으로 검사할 수 있다는 것입니다. 이는 다음에 중요합니다:

**멀티 에이전트 시스템 디버깅:**
- 각 에이전트가 무엇을 논의했는지 정확히 확인
- 대화의 어떤 부분을 어떤 에이전트가 처리했는지 식별
- 시스템을 통한 정보 흐름 추적

**병렬 실행 이해:**
- 에이전트가 별도의 컨텍스트를 유지했는지 확인
- 동시 실행 중 메모리 충돌이 없었는지 확인
- 에이전트 상호작용의 타임라인 감사

대화 중 생성된 브랜치를 살펴보겠습니다:

In [ ]:
print("\n=== 메모리 브랜치 조회 ===")

if flight_memory_hooks or hotel_memory_hooks:
    # 브랜치를 나열하기 위해 아무 메모리 세션이나 가져옴 (모두 같은 세션을 가리킴)
    hook = flight_memory_hooks if flight_memory_hooks else hotel_memory_hooks
    if hook:
        memory_session = hook.get_session(actor_id, session_id)

        # 세션의 모든 브랜치 나열
        branches = memory_session.list_branches()
        print(f"\n세션에 총 {len(branches)}개의 브랜치가 있습니다:")
        for branch in branches:
            print(f"  - 브랜치: {branch.name}")
            print(f"    └─ 이벤트: {len(memory_session.list_events(branch_name=branch.name))}")
            print(f"    └─ 생성일: {branch.created}")

        print("\n각 브랜치는 다른 에이전트의 메모리를 나타냅니다:")
        print("  • 'main' = 여행 코디네이터 대화")
        print("  • 'flight_agent_memory' = 항공편 어시스턴트 대화")
        print("  • 'hotel_agent_memory' = 호텔 어시스턴트 대화")

### 브랜치별 대화 기록 접근

이제 각 브랜치에 저장된 실제 대화를 자세히 살펴보겠습니다. 이는 메모리 브랜칭이 공유 세션을 유지하면서 에이전트 간 완전한 격리를 제공하는 방법을 보여줍니다.

In [ ]:
print("\n=== 브랜치별 이벤트 접근 ===")

if flight_memory_hooks or hotel_memory_hooks:
    hook = flight_memory_hooks if flight_memory_hooks else hotel_memory_hooks
    if hook:
        memory_session = hook.get_session(actor_id, session_id)

        # main 브랜치 (코디네이터)의 이벤트 가져오기
        main_events = memory_session.list_events(branch_name="main")
        print(f"\nMain 브랜치 - 코디네이터 ({len(main_events)}개 이벤트):")
        if main_events:
            for event in main_events[-3:]:  # 마지막 3개 이벤트 표시
                for payload in event.payload:
                    if 'conversational' in payload:
                        role = payload['conversational']['role']
                        text = payload['conversational']['content']['text']
                        print(f"  {role}: {text[:100]}...")
        else:
            print("  main 브랜치에 이벤트가 없습니다")

        # 항공편 에이전트 브랜치의 이벤트 가져오기
        try:
            flight_branch_events = memory_session.list_events(branch_name="flight_agent_memory")
            print(f"\n항공편 에이전트 브랜치 ({len(flight_branch_events)}개 이벤트):")
            if flight_branch_events:
                print("모든 항공편 관련 대화가 여기에 저장됩니다:")
                for event in flight_branch_events[-3:]:  # 마지막 3개 이벤트 표시
                    for payload in event.payload:
                        if 'conversational' in payload:
                            role = payload['conversational']['role']
                            text = payload['conversational']['content']['text']
                            print(f"  {role}: {text[:100]}...")
            else:
                print("  이벤트 없음 - 항공편 어시스턴트가 아직 호출되지 않았습니다")
        except Exception as e:
            print(f"  항공편 브랜치가 아직 생성되지 않았습니다: {e}")

        # 호텔 에이전트 브랜치의 이벤트 가져오기
        try:
            hotel_branch_events = memory_session.list_events(branch_name="hotel_agent_memory")
            print(f"\n호텔 에이전트 브랜치 ({len(hotel_branch_events)}개 이벤트):")
            if hotel_branch_events:
                print("모든 호텔 관련 대화가 여기에 저장됩니다:")
                for event in hotel_branch_events[-3:]:  # 마지막 3개 이벤트 표시
                    for payload in event.payload:
                        if 'conversational' in payload:
                            role = payload['conversational']['role']
                            text = payload['conversational']['content']['text']
                            print(f"  {role}: {text[:100]}...")
            else:
                print("  이벤트 없음 - 호텔 어시스턴트가 아직 호출되지 않았습니다")
        except Exception as e:
            print(f"  호텔 브랜치가 아직 생성되지 않았습니다: {e}")

## 요약

이 노트북에서 **AgentCore 메모리 브랜칭** - 병렬 실행이 가능한 견고한 멀티 에이전트 시스템 구축을 위한 핵심 기능을 보여주었습니다:

### 주요 요점:

1. **메모리 브랜칭이 병렬 실행을 가능하게 함**
   - 여러 에이전트가 메모리 충돌 없이 동시에 실행 가능
   - 각 에이전트가 브랜치를 통해 자체 대화 컨텍스트를 유지
   - Strands Agent Graph 및 기타 병렬 에이전트 프레임워크에 필수적

2. **컨텍스트 격리가 에이전트 성능을 향상**
   - 전문 에이전트가 간섭 없이 자신의 도메인에 집중
   - 에이전트 간 컨텍스트 오염 없음
   - 더 깔끔하고 관련성 높은 대화

3. **격리된 컨텍스트를 가진 공유 세션**
   - 단일 메모리 리소스 및 세션 ID
   - 다른 에이전트를 위한 여러 브랜치
   - 효율적인 리소스 활용


### 아키텍처 패턴:

```
Memory Resource (memory_id)
  └── Session (session_id)
      ├── main branch → 코디네이터 에이전트
      ├── flight_agent_memory → 항공편 에이전트 (병렬 실행 가능)
      └── hotel_agent_memory → 호텔 에이전트 (병렬 실행 가능)
```

AgentCore 메모리 브랜칭은 애플리케이션 요구에 맞게 확장할 수 있는 정교한 멀티 에이전트 시스템을 안전하고 효율적으로 구축할 수 있게 합니다.

## 정리
이 노트북에서 사용한 리소스를 정리하기 위해 메모리를 삭제하겠습니다.

In [ ]:
#client.delete_memory_and_wait(
#        memory_id = memory_id,
#        max_wait = 300,
#        poll_interval =10
#)